# SI4006 · Sesión 3 — Lab: **El bloque y las familias**

**Tópicos Especiales y Aplicaciones en IA** · Universidad EAFIT · Módulo 1 — Transformers

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

La semana pasada vieron cómo un transformer tokeniza, mira y presta atención.
Hoy **armamos el bloque completo**, lo **rompemos a propósito** para entender por qué está hecho así,
y ponemos a las **tres familias** (encoder, decoder, encoder-decoder) a resolver el *mismo* problema
para que puedan decidir cuál le sirve a su proyecto.

> **Cómo se usa este notebook.** Vamos a saltar entre estas celdas y las slides. Cada vez que vean
> **🔄 PUNTO DE SINCRONIZACIÓN**, paren, corran lo que va hasta ahí y volvemos juntos al proyector.
> Los recuadros con 🙋 son preguntas; los ✍️ son suyos, para escribir código.

## 0 · Setup

Primero miramos qué trae Colab. **Novedad 2026:** ya no fijamos la versión de `transformers` —
Colab viene con una versión reciente (5.x) y forzar una vieja rompe otras dependencias. Corremos con
lo que ya está instalado; las APIs que usamos hoy son estables entre versiones.

In [1]:
# Diagnóstico: qué trae este Colab
import importlib.metadata as m
for pkg in ['torch','transformers','sentencepiece']:
    try: print(f'{pkg:<15}', m.version(pkg))
    except Exception: print(f'{pkg:<15} (no instalado)')

torch           2.11.0+cpu
transformers    5.13.1
sentencepiece   0.2.2


In [2]:
# Único install: aseguramos sentencepiece (lo pide el tokenizer de T5).
# NO fijamos transformers/torch: usamos los que ya trae Colab -> sin conflictos ni restart.
%pip install -q sentencepiece
print('\nListo. Usamos los paquetes que ya trae Colab.')


Listo. Usamos los paquetes que ya trae Colab.


In [3]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device:', device)
# En Colab gratis lo normal es CPU. Todo lo de hoy corre en CPU; solo tarda un poco más.

torch 2.11.0+cpu | device: cpu


---

# LAB A · El bloque, armado y roto

En las slides está la anatomía de un bloque: **Multi-Head Attention → Add & Norm → FFN → Add & Norm**.
Aquí lo construimos en PyTorch, con un interruptor para **activar o quitar la conexión residual** — ese
`x +` que parece un detalle y en realidad es lo que deja entrenar modelos profundos.

### A.1 · El bloque, para completar  ✍️

Completen el bloque:

> **Un detalle de diseño.** Usamos la variante **pre-norm** (LayerNorm *antes* de cada sub-bloque), que
> es la que usan los modelos modernos. El paper de 2017 la ponía después (*post-norm*). Con pre-norm el
> residual queda como un atajo limpio (`x + f(...)`), y por eso el efecto sobre el gradiente se ve nítido.

In [4]:
import torch.nn as nn

class BloqueTransformer(nn.Module):
    def __init__(self, d=128, n_heads=4, usar_residual=True):
        super().__init__()
        self.usar_residual = usar_residual
        self.attn = nn.MultiheadAttention(d, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))  # expande 4x, no-lineal, proyecta
        self.norm2 = nn.LayerNorm(d)

    def _residual(self, entrada, salida_subbloque):
        # 'entrada' = x que entró al sub-bloque; 'salida_subbloque' = f(x).
        if self.usar_residual:
            # TODO ✍️ — devuelvan la CONEXIÓN RESIDUAL: la entrada + la salida del sub-bloque.
            return salida_subbloque
        else:
            return salida_subbloque            # sin residual: se pierde el atajo del gradiente

    def forward(self, x):
        h = self.norm1(x)                      # pre-norm
        a, _ = self.attn(h, h, h)              # self-attention
        x = self._residual(x, a)               # atajo alrededor de la atención
        f = self.ffn(self.norm2(x))            # pre-norm + FFN
        x = self._residual(x, f)               # atajo alrededor de la FFN
        return x

print('Bloque definido. Un bloque =', sum(p.numel() for p in BloqueTransformer().parameters()), 'parámetros.')

Bloque definido. Un bloque = 198272 parámetros.


<details><summary>🔑 Solución — ábranla solo si llevan varios minutos atascados</summary>

```python
return entrada + salida_subbloque
```

Eso es todo. `x + f(x)` en vez de `f(x)`. Esa suma es la "autopista" por la que el gradiente vuelve
hasta las primeras capas casi intacto.
</details>

### A.2 · Lo apilamos 48 veces y medimos el gradiente

Un modelo real no es un bloque: son decenas apilados. Vamos a construir **una torre de 48 bloques**,
meterle un tensor, calcular una pérdida cualquiera y ver **con qué fuerza llega el gradiente hasta la
primera capa**. Ese número — la norma del gradiente en la capa 0 — es el termómetro de si la red
profunda puede aprender.

In [5]:
def norma_gradiente_capa0(usar_residual, n_capas=48, d=128, seeds=range(5)):
    # Promediamos sobre varias inicializaciones para que el resultado no dependa del azar de una sola.
    normas = []
    for sd in seeds:
        torch.manual_seed(sd)
        torre = nn.Sequential(*[BloqueTransformer(d=d, usar_residual=usar_residual) for _ in range(n_capas)])
        x = torch.randn(1, 16, d, requires_grad=True)   # (batch, tokens, dim)
        torre(x).pow(2).mean().backward()               # una pérdida cualquiera para tener gradientes
        g = torch.cat([p.grad.flatten() for p in torre[0].parameters() if p.grad is not None])
        normas.append(g.norm().item())
    return sum(normas) / len(normas)

con = norma_gradiente_capa0(usar_residual=True)
sin = norma_gradiente_capa0(usar_residual=False)
print(f'Norma del gradiente en la capa 0 (torre de 48)  ·  CON residual: {con:.3e}')
print(f'Norma del gradiente en la capa 0 (torre de 48)  ·  SIN residual: {sin:.3e}')
print(f'\nCon residual, la señal que llega a la primera capa es ~{con/sin:.0f}x más fuerte.')

Norma del gradiente en la capa 0 (torre de 48)  ·  CON residual: 7.020e-01
Norma del gradiente en la capa 0 (torre de 48)  ·  SIN residual: 7.020e-01

Con residual, la señal que llega a la primera capa es ~1x más fuerte.


> 🙋 ¿Cuál de las dos torres va a aprender en sus capas bajas y cuál no?

<details><summary>🔑 Respuesta </summary>

Si el gradiente que llega a la capa 0 es casi cero, esos pesos **no se mueven** durante el entrenamiento: la profundidad
se desperdicia. Este es, otra vez, el **vanishing gradient** de Redes Neuronales, esta conexión
residual es lo que hizo posible entrenar ResNet y, después, los transformers profundos.

### A.3 · ¿Dónde viven los parámetros?

En las slides dijimos que la FFN pesa más que la atención. Compruébenlo.

In [6]:
b = BloqueTransformer()
p_attn = sum(p.numel() for p in b.attn.parameters())
p_ffn  = sum(p.numel() for p in b.ffn.parameters())
print(f'Attention : {p_attn:>8,} parámetros')
print(f'FFN       : {p_ffn:>8,} parámetros')
print(f'\nLa FFN tiene {p_ffn/p_attn:.1f}x los parámetros de la atención. Los pesos viven, sobre todo, en la FFN.')

Attention :   66,048 parámetros
FFN       :  131,712 parámetros

La FFN tiene 2.0x los parámetros de la atención. Los pesos viven, sobre todo, en la FFN.


### 🔄 PUNTO DE SINCRONIZACIÓN — volvemos a las slides

Ya midieron dos cosas: **el residual salva el gradiente** y **la FFN carga con
los parámetros**. Ahora las *tres familias*.

---

# LAB B · Tres familias, un mismo problema

Cargamos un representante de cada familia y les damos, hasta donde se pueda, **la misma frase**.
La idea no es que uno sea "mejor": es ver que **cada familia hace un trabajo distinto** porque está
cableada distinto.

| Familia | Modelo de hoy | Tamaño | Para qué es bueno |
|---|---|---|---|
| Encoder-only | `distilbert-base-multilingual-cased` | ~135M | entender / clasificar / rellenar |
| Decoder-only | `Qwen/Qwen2.5-0.5B-Instruct` | ~0.5B | generar / conversar |
| Encoder-decoder | `t5-small` | ~60M | transformar texto → texto |

In [7]:
# Descarga ~2 min la primera vez. Los tres son pequeños y caben de sobra en Colab gratis.
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          AutoModelForSeq2SeqLM, pipeline)

# --- Encoder: BERT (rellena un hueco) ---
bert = pipeline('fill-mask', model='distilbert-base-multilingual-cased')

# --- Decoder: Qwen (genera) ---
qwen_tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
qwen = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')  # float32 por defecto en CPU

# --- Encoder-decoder: T5 base (transforma; y hace 'span corruption', que veremos en el Lab C) ---
t5_tok = AutoTokenizer.from_pretrained('t5-small')
t5 = AutoModelForSeq2SeqLM.from_pretrained('t5-small')
print('Los tres modelos, cargados.')

config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Los tres modelos, cargados.


### B.1 · Encoder — BERT llena el hueco

El encoder mira **toda** la frase a la vez. Le tapamos una palabra y le pedimos que la adivine con el
contexto de ambos lados.

In [8]:
frase = 'Roma es la capital de [MASK].'
for r in bert(frase, top_k=3):
    print(f"{r['token_str']:<12} (confianza {r['score']:.2f})")

Italia       (confianza 0.59)
Roma         (confianza 0.09)
స్టేషన్      (confianza 0.08)


> 👀 BERT **entiende** el contexto y llena el hueco. Lo que **no** hace es escribir una respuesta libre
> desde cero: no fue entrenado para generar, sino para comprender.

### B.2 · Decoder — Qwen continúa la frase

El decoder solo puede mirar **hacia atrás** (máscara causal). Le damos un principio y **genera** lo que sigue.

In [9]:
mensajes = [{'role':'user','content':'Completa en una frase: La capital de Francia es'}]
# En transformers 5.x, apply_chat_template con return_dict=True devuelve un dict de tensores;
# se lo pasamos a generate con **inputs (patrón estable en 4.x y 5.x).
inputs = qwen_tok.apply_chat_template(mensajes, add_generation_prompt=True,
                                      return_tensors='pt', return_dict=True)
salida = qwen.generate(**inputs, max_new_tokens=30, do_sample=False)
print(qwen_tok.decode(salida[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

La capital de Francia es París.


> ✍️ **Genera texto fluido** — pero es un martillo distinto. Pedirle que "clasifique" es forzarlo a
> hablar de una etiqueta en vez de simplemente devolverla.

### B.3 · Encoder-decoder — T5 transforma el texto

Lee toda la entrada (encoder) y **produce** una salida transformada (decoder). Es la familia natural para
traducir, resumir o reescribir. T5 se maneja con un *prefijo* que dice qué tarea hacer.

In [10]:
prompt = 'translate English to French: The cat is black.'
ids = t5_tok(prompt, return_tensors='pt').input_ids
out = t5.generate(ids, max_new_tokens=20)
print(t5_tok.decode(out[0], skip_special_tokens=True))

Le chat est noir.


### B.4 · Su dominio  ✍️

Ahora ustedes. Cambien la frase por algo **de su proyecto** y pásenla por la familia que crean más natural.
Un consejo: si su tarea es *rellenar/clasificar*, prueben BERT; si es *generar/conversar*, Qwen; si es
*transformar un texto en otro*, T5.

In [16]:
# TODO ✍️ — reemplacen por texto REAL de su dominio y descomenten SU familia.

# --- Opción A · Encoder (BERT): rellenar / clasificar -------------------
# Pongan [MASK] donde quieren que el modelo adivine.
mi_frase = 'La inteligencia [MASK] está transformando la industria.'
for r in bert(mi_frase, top_k=3):
    print(r['token_str'], round(r['score'], 3))

# --- Opción B · Decoder (Qwen): generar / conversar --------------------
# mi_prompt = 'Escribe una disculpa breve a un cliente por una entrega tardía:'
# print(qwen(mi_prompt, max_new_tokens=40)[0]['generated_text'])

# --- Opción C · Encoder-decoder (T5): transformar texto → texto --------
# mi_entrada = 'translate English to French: The cat is black'
# print(t5(mi_entrada)[0]['generated_text'])

artificial 0.053
internacional 0.041
economica 0.036


> 🙋 ¿Qué familia les encaja mejor y por qué?

### 🔄 PUNTO DE SINCRONIZACIÓN — volvemos a las slides

Misma entrada, tres salidas distintas. Volvemos al proyector para ver **de dónde** sale esa diferencia:
el objetivo con el que cada familia fue pre-entrenada.

---

# LAB C · El objetivo, en vivo

Cada familia es buena en algo porque fue **entrenada con un objetivo distinto**. Aquí vemos ese objetivo
hecho código, con los mismos tres modelos.

### C.1 · MLM — Masked Language Modeling (encoder)

El objetivo de BERT: tapar tokens al azar y adivinarlos usando **ambos lados**. Por eso entiende contexto.
Miren los candidatos y su probabilidad.

In [12]:
for r in bert('El modelo predice el token que fue [MASK] durante el entrenamiento.', top_k=5):
    print(f"{r['token_str']:<14} {r['score']:.3f}")

utilizado      0.169
usado          0.151
desarrollado   0.125
lanzado        0.095
賣              0.051


### C.2 · CLM — Causal Language Modeling (decoder)

El objetivo de Qwen: dado lo anterior, **¿cuál es el siguiente token?** No adivina un hueco: predice un paso a la vez.

In [13]:
import torch.nn.functional as F
ctx = 'Los planetas giran alrededor del'
ids = qwen_tok(ctx, return_tensors='pt').input_ids
with torch.no_grad():
    logits = qwen(ids).logits[0, -1]          # logits del SIGUIENTE token
probs = F.softmax(logits, dim=-1)
top = torch.topk(probs, 8)
print(f'Contexto: "{ctx} ..."\nCandidatos al siguiente token:')
for p, i in zip(top.values, top.indices):
    print(f"  {qwen_tok.decode(i)!r:<16} {p.item():.3f}")

Contexto: "Los planetas giran alrededor del ..."
Candidatos al siguiente token:
  ' Sol'           0.750
  ' sol'           0.147
  ' centro'        0.014
  ' Sistema'       0.011
  ' mismo'         0.008
  ' plan'          0.008
  ' sistema'       0.007
  ' objeto'        0.002


### C.3 · Span corruption — el objetivo de T5 (encoder-decoder)

El objetivo con el que T5 fue pre-entrenado: **borrar tramos** del texto y reconstruirlos. El hueco se
marca con un centinela `<extra_id_0>`, y el modelo genera lo que iba ahí — con su propio centinela delante.
Por eso decodificamos **sin** quitar los tokens especiales: para ver el mecanismo tal cual.

In [14]:
prompt = 'The sky is usually <extra_id_0> during the day.'
ids = t5_tok(prompt, return_tensors='pt').input_ids
out = t5.generate(ids, max_new_tokens=8)
print('Salida cruda :', t5_tok.decode(out[0], skip_special_tokens=False).replace('<pad>','').strip())
print('El relleno   :', t5_tok.decode(out[0], skip_special_tokens=True))

Salida cruda : <extra_id_0> blue<extra_id_1> blue.</s>
El relleno   : blue blue.


> 🧩 **El objetivo moldea el talento**:

> | Objetivo | Familia | Talento que deja |
> |---|---|---|
> | MLM (adivinar huecos) | Encoder | comprender |
> | CLM (siguiente token) | Decoder | generar |
> | Span corruption | Encoder-decoder | transformar |

### 🔄 PUNTO DE SINCRONIZACIÓN — volvemos a las slides

Con esto conectamos objetivo → familia → para qué sirve. Volvemos al proyector para cerrar con **la
decisión que se llevan de hoy**

---

# Cierre · La decisión para M1

En S04 van a **fine-tunear** un modelo base para su proyecto. No empiecen de la página en blanco: ya saben que familia y un modelo candidato les puede ser util. Esta pequeña ayuda traduce *qué hace su sistema* en *qué
familia arrancar*.

In [15]:
def sugerir_familia(que_hace_su_sistema):
    t = que_hace_su_sistema.lower()
    if any(k in t for k in ['clasific','detect','extra','etiqu','sentimiento','spam','categor']):
        return 'Encoder  → prueben DistilBERT / BERT multilingüe'
    if any(k in t for k in ['gener','chat','conversa','escrib','redact','responde','asistente']):
        return 'Decoder  → prueben Qwen2.5-0.5B / TinyLlama'
    if any(k in t for k in ['traduc','resum','reescrib','transform','corrig','parafr']):
        return 'Encoder-decoder → prueben T5-small / FlanT5-small / BART'
    return 'No es evidente: descríbanlo como clasificar / generar / transformar y vuelvan a intentar.'

# TODO ✍️ — describan en una frase qué hace su sistema:
mi_sistema = 'clasificar reseñas de clientes como positivas o negativas'
print('Familia sugerida:', sugerir_familia(mi_sistema))

Familia sugerida: Encoder  → prueben DistilBERT / BERT multilingüe


> ✍️
> **familia elegida + un modelo del Hub** ([huggingface.co/models](https://huggingface.co/models)) para
> arrancar.

---

*SI4006 · Universidad EAFIT · Sesión 3 — El bloque y las familias.*